# Final Test Run: Models Refitted on Train + Validation

## Natural-language request

> Retrain CatBoost and XGBoost on training and validation data both, in another notebook, and then do the final test run.

This is **run 3** on the proof period (arrivals 1 January to 31 August 2017), with settings frozen in notebook 15 (`models/frozen_choice_run3_train_val.json`) **before** this notebook opened test.

| | Run 3 setting |
|---|---|
| Primary method | **CatBoost**, refitted on train + val |
| Also reported | **XGBoost**, refitted on train + val (not selected on these results) |
| Rule and buffers | k = max(0, ⌊expected⌋ − buffer); **City 6, Resort 3** for both models (chosen on val) |
| Benchmark | Flat rule with **train** rates (as the contract specifies), buffers **City 20, Resort 4** |
| Fairness check | Flat rule with **train + val** rates, same buffers, because the models saw val's data |
| Guardrail | At most **10 walks per 1,000 arrivals**, group-wide |
| Secondary | Shared buffers (models 5, flat rule 16), disclosed as flattering the models |

## Disclosure (must accompany every result here)

Run 3 was decided **after** run 1's test result was seen (CatBoost fitted on train only: EUR 150,459 more than the flat rule). Run 2 (XGBoost fitted on train only) was frozen but never executed. **All runs are reported together**, in Step 8 and in `docs/DECISIONS.md`. Run 3 is therefore not a clean first-look proof result on its own.

The order is enforced as in notebook 14: check the fingerprints, predict and save before opening the labels, then score.

## Step 1: check the frozen settings

In [1]:
from pathlib import Path
import hashlib
import json
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score

# Find the repository so this works from the repo folder or notebooks/.
repo_root = None
for folder in (Path.cwd(), *Path.cwd().parents):
    if (folder / "src" / "contract.py").exists():
        repo_root = folder
        break
if repo_root is None:
    raise FileNotFoundError("Could not find the project src/ folder.")
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import contract


def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


freeze_path = repo_root / "models" / "frozen_choice_run3_train_val.json"
freeze = json.loads(freeze_path.read_text(encoding="utf-8"))
freeze_fingerprint = sha256(freeze_path)
assert freeze["run"] == 3 and freeze["frozen_before_test"]
assert sha256(repo_root / freeze["preprocessor_file"]) == freeze["preprocessor_sha256"], "preprocessor changed after the freeze"
for name, info in freeze["models"].items():
    assert sha256(repo_root / info["file"]) == info["sha256"], f"{name} changed after the freeze"

model_dir = repo_root / "data" / "models"
run_record_path = model_dir / "test_run3_record.json"
if run_record_path.exists() and json.loads(run_record_path.read_text(encoding="utf-8"))["freeze_sha256"] != freeze_fingerprint:
    raise RuntimeError("Run 3's frozen settings changed after it was run. Disclose the change and report every run.")
run1_record = json.loads((model_dir / "test_run_record.json").read_text(encoding="utf-8"))

split_dir = repo_root / "data" / "splits"
manifest = json.loads((split_dir / "split_columns.json").read_text(encoding="utf-8"))
inputs, label = manifest["inputs"], manifest["label"]
HOTELS = ["City Hotel", "Resort Hotel"]
PRIMARY = freeze["primary_method"]
MODELS = [PRIMARY, *freeze["also_reported"]]
FLAT, FLAT_FAIR = "flat rule (train rates)", "flat rule (train + val rates)"
print(f"Run 3 frozen at {freeze['frozen_at']}; fingerprints match. Primary: {PRIMARY}; also reported: {freeze['also_reported']}.")

Run 3 frozen at 2026-11-18T02:05:41; fingerprints match. Primary: CatBoost; also reported: ['XGBoost'].


## Step 2: predict from the test inputs, before opening any outcome

In [2]:
test = pd.read_csv(split_dir / "test.csv", index_col="row", parse_dates=["arrival_date"], dtype={"agent": str, "company": str})
assert label not in test.columns and test["arrival_date"].min() >= pd.Timestamp(contract.PROOF_START)

preprocessor = joblib.load(repo_root / freeze["preprocessor_file"])
models = {name: joblib.load(repo_root / freeze["models"][name]["file"]) for name in MODELS}
before = [joblib.hash(preprocessor), *(joblib.hash(m) for m in models.values())]

warnings.filterwarnings("ignore", message="Found unknown categories")
X_test = preprocessor.transform(test[inputs]).to_numpy(dtype=float)
predictions = pd.DataFrame({name: m.predict_proba(X_test)[:, 1] for name, m in models.items()}, index=test.index)
predictions[FLAT] = test["hotel"].map(freeze["flat_rule_hotel_rates_from_train"]).to_numpy()
predictions[FLAT_FAIR] = test["hotel"].map(freeze["fairness_check_flat_rule_rates_from_train_and_choose"]).to_numpy()

assert before == [joblib.hash(preprocessor), *(joblib.hash(m) for m in models.values())], "nothing may be refitted on test"
assert predictions.notna().all().all() and predictions.apply(lambda p: p.between(0, 1).all()).all()
predictions_path = model_dir / "test_run3_predictions.csv"
predictions.to_csv(predictions_path, index_label="row")
predictions_sha = sha256(predictions_path)
print(f"Predicted {len(test):,} test bookings; predictions saved before any outcome was opened.")

Predicted 40,687 test bookings; predictions saved before any outcome was opened.


## Step 3: open the test labels

In [3]:
assert sha256(predictions_path) == predictions_sha
test_labels = pd.read_csv(split_dir / "test_labels.csv", index_col="row")
assert test_labels.index.equals(test.index)
test = test.join(test_labels)
y_test = test[label]
print(f"Test: {len(test):,} bookings, {test.groupby(['hotel', 'arrival_date']).ngroups} hotel-nights, "
      f"actual cancellation rate {y_test.mean():.1%}")

Test: 40,687 bookings, 486 hotel-nights, actual cancellation rate 38.7%


## Step 4: accuracy of the booking-level forecasts

Accuracy at a 50% cut-off, with the reference of predicting "nobody cancels", plus ROC AUC, log loss, Brier score and the mean predicted rate. Run 1's CatBoost (train-only fit) is included for comparison.

In [4]:
run1_predictions = pd.read_csv(model_dir / "test_predictions.csv", index_col="row").loc[test.index]
columns = {**{n: predictions[n] for n in MODELS}, "CatBoost, run 1 (train only)": run1_predictions["CatBoost"],
           FLAT: predictions[FLAT], FLAT_FAIR: predictions[FLAT_FAIR]}
accuracy_table = pd.DataFrame({
    name: {"accuracy (50% cut-off)": accuracy_score(y_test, p >= 0.5), "ROC AUC": roc_auc_score(y_test, p),
           "log loss": log_loss(y_test, p), "Brier score": brier_score_loss(y_test, p),
           "mean predicted rate": p.mean(), "actual rate": y_test.mean()}
    for name, p in columns.items()
}).T
print(f"Accuracy of predicting 'nobody cancels': {1 - y_test.mean():.1%}")
display(accuracy_table.style.format({"accuracy (50% cut-off)": "{:.1%}", "ROC AUC": "{:.3f}", "log loss": "{:.4f}",
                                     "Brier score": "{:.4f}", "mean predicted rate": "{:.1%}", "actual rate": "{:.1%}"}))

Accuracy of predicting 'nobody cancels': 61.3%


,accuracy (50% cut-off),ROC AUC,log loss,Brier score,mean predicted rate,actual rate
CatBoost,63.9%,0.687,0.6188,0.2167,31.0%,38.7%
XGBoost,63.5%,0.690,0.6183,0.2166,31.0%,38.7%
"CatBoost, run 1 (train only)",63.5%,0.679,0.6427,0.2270,27.2%,38.7%
flat rule (train rates),61.3%,0.554,0.6636,0.2353,36.3%,38.7%
flat rule (train + val rates),61.3%,0.554,0.6627,0.2349,36.5%,38.7%


## Step 5: expected vs actual cancellations per hotel-night

Bias above zero means too many cancellations were expected (a walk risk at a fixed buffer), below zero too few (an empty-room risk).

In [5]:
night_tables = {m: contract.nightly_table(test[contract.REQUIRED], predictions[m]) for m in predictions}
rows = []
for name, table in night_tables.items():
    error = table["expected"] - table["cancelled"]
    for hotel in ["group", *HOTELS]:
        e = error if hotel == "group" else error.xs(hotel, level="hotel")
        rows.append({"hotel": hotel, "method": name, "bias (rooms)": e.mean(), "mean absolute error (rooms)": e.abs().mean(),
                     "spread (rooms)": e.std()})
display(pd.DataFrame(rows).set_index(["hotel", "method"]).style.format("{:+.2f}", subset=["bias (rooms)"])
        .format("{:.2f}", subset=["mean absolute error (rooms)", "spread (rooms)"]))

,,bias (rooms),mean absolute error (rooms),spread (rooms)
hotel,method,,,
group,CatBoost,-6.43,9.61,13.75
City Hotel,CatBoost,-9.02,13.01,17.69
Resort Hotel,CatBoost,-3.84,6.20,7.25
group,XGBoost,-6.43,9.60,13.67
City Hotel,XGBoost,-8.98,13.01,17.60
Resort Hotel,XGBoost,-3.88,6.19,7.20
group,flat rule (train rates),-2.00,9.31,14.32
City Hotel,flat rule (train rates),-0.95,12.29,18.67
Resort Hotel,flat rule (train rates),-3.04,6.33,7.77


## Step 6: cost, saving and guardrail at the frozen buffers

Euros, empty rooms, walked guests and walks per 1,000, side by side (project rule 5). **Saving** is the benchmark flat rule's cost (train rates) minus each method's cost, on the same 486 hotel-nights. **Guardrail** is the group-wide ceiling; hotel rates are reported without a threshold (project rule 6).

In [6]:
def score(method, buffers):
    table = night_tables[method]
    buffer = np.where(table.index.get_level_values("hotel") == "City Hotel", buffers["City Hotel"], buffers["Resort Hotel"])
    k = pd.Series(np.maximum(np.floor(table["expected"].to_numpy()) - buffer, 0), index=table.index)
    out = {"group": contract.night_cost(table, k)}
    for hotel in HOTELS:
        part = table.xs(hotel, level="hotel", drop_level=False)
        out[hotel] = contract.night_cost(part, k.loc[part.index])
    return out


def cost_table(buffers_by_method):
    results = {m: score(m, b) for m, b in buffers_by_method.items()}
    rows = []
    for hotel in ["group", *HOTELS]:
        for m, r in results.items():
            r = r[hotel]
            rows.append({"hotel": hotel, "method": m,
                         "buffer": buffers_by_method[m][hotel] if hotel != "group" else "",
                         "cost (EUR)": r["cost_eur"], "empty rooms": r["empty_rooms"], "walked guests": r["walked_guests"],
                         "walks per 1,000": r["walks_per_1000"],
                         "guardrail (group <= 10)": ("met" if r["ceiling_met"] else "BREACHED") if hotel == "group" else "n/a",
                         "saving vs flat rule (EUR)": results[FLAT][hotel]["cost_eur"] - r["cost_eur"],
                         "saving vs flat rule (%)": 1 - r["cost_eur"] / results[FLAT][hotel]["cost_eur"]})
    return pd.DataFrame(rows).set_index(["hotel", "method"]), results


FMT = {"cost (EUR)": "{:,.0f}", "walks per 1,000": "{:.2f}", "saving vs flat rule (EUR)": "{:+,.0f}", "saving vs flat rule (%)": "{:+.1%}"}
primary_buffers = {**freeze["primary"]["model_buffers"], FLAT: freeze["primary"]["flat_rule_buffers"],
                   FLAT_FAIR: freeze["primary"]["flat_rule_buffers"]}
primary_table, primary_results = cost_table(primary_buffers)
print("PRIMARY: separate buffer per hotel (frozen)")
display(primary_table.style.format(FMT))

both = lambda b: {"City Hotel": b, "Resort Hotel": b}
sec = freeze["secondary_reported_for_transparency"]
secondary_buffers = {**{m: both(b) for m, b in sec["model_buffers"].items()}, FLAT: both(sec["flat_rule_buffer"]),
                     FLAT_FAIR: both(sec["flat_rule_buffer"])}
secondary_table, secondary_results = cost_table(secondary_buffers)
print("SECONDARY (disclosed): one shared buffer, which handicaps the flat rule")
display(secondary_table.style.format(FMT))

PRIMARY: separate buffer per hotel (frozen)


SECONDARY (disclosed): one shared buffer, which handicaps the flat rule


## Step 7: month by month (primary settings)

In [7]:
rows = []
for m in [FLAT, *MODELS]:
    table = night_tables[m]
    b = primary_buffers[m]
    buffer = np.where(table.index.get_level_values("hotel") == "City Hotel", b["City Hotel"], b["Resort Hotel"])
    k = np.maximum(np.floor(table["expected"].to_numpy()) - buffer, 0)
    walks = (k - table["cancelled"]).clip(lower=0)
    cost = (table["cancelled"] - k).clip(lower=0) * table["stay_value"] + walks * contract.WALK_COST
    month = table.index.get_level_values("arrival_date").to_period("M").astype(str)
    arrivals = (table["book"] - table["cancelled"]).groupby(month).sum()
    for mo, c in cost.groupby(month).sum().items():
        rows.append({"month": mo, "method": m, "cost (EUR)": c, "walks per 1,000": 1000 * walks.groupby(month).sum()[mo] / arrivals[mo]})
monthly = pd.DataFrame(rows).pivot(index="month", columns="method")
display(monthly.style.format("{:,.0f}", subset=[c for c in monthly.columns if c[0] == "cost (EUR)"])
        .format("{:.2f}", subset=[c for c in monthly.columns if c[0] == "walks per 1,000"]))
for m in MODELS:
    wins = (monthly[("cost (EUR)", m)] < monthly[("cost (EUR)", FLAT)]).sum()
    print(f"{m} cheaper than the flat rule in {wins} of {len(monthly)} months.")

CatBoost cheaper than the flat rule in 8 of 8 months.
XGBoost cheaper than the flat rule in 8 of 8 months.


## Step 8: every run, side by side

Required because run 3 was decided after run 1's result (`docs/EVALUATION_CONTRACT.md`). All rows use the same benchmark: the flat rule with train rates and per-hotel buffers City 20 / Resort 4.

In [8]:
run1 = run1_record["primary"]
rows = [{"run": "1 (frozen before any test result)", "method": "CatBoost, fitted on train", "cost (EUR)": run1["catboost"]["cost_eur"],
         "empty rooms": run1["catboost"]["empty_rooms"], "walked guests": run1["catboost"]["walked_guests"],
         "walks per 1,000": run1["catboost"]["walks_per_1000"], "guardrail": "met" if run1["catboost"]["ceiling_met"] else "BREACHED",
         "saving vs flat rule (EUR)": run1["saving_eur"]},
        {"run": "2", "method": "XGBoost, fitted on train", "cost (EUR)": np.nan, "empty rooms": np.nan, "walked guests": np.nan,
         "walks per 1,000": np.nan, "guardrail": "not run (superseded)", "saving vs flat rule (EUR)": np.nan}]
for m in MODELS:
    r = primary_results[m]["group"]
    rows.append({"run": "3 (after run 1's result)" + (", primary" if m == PRIMARY else ", also reported"),
                 "method": f"{m}, refitted on train + val", "cost (EUR)": r["cost_eur"], "empty rooms": r["empty_rooms"],
                 "walked guests": r["walked_guests"], "walks per 1,000": r["walks_per_1000"],
                 "guardrail": "met" if r["ceiling_met"] else "BREACHED",
                 "saving vs flat rule (EUR)": primary_results[FLAT]["group"]["cost_eur"] - r["cost_eur"]})
assert abs(primary_results[FLAT]["group"]["cost_eur"] - run1["flat_rule"]["cost_eur"]) < 0.01, "the benchmark must be identical across runs"
all_runs = pd.DataFrame(rows).set_index(["run", "method"])
display(all_runs.style.format({"cost (EUR)": "{:,.0f}", "empty rooms": "{:,.0f}", "walked guests": "{:,.0f}",
                               "walks per 1,000": "{:.2f}", "saving vs flat rule (EUR)": "{:+,.0f}"}, na_rep="-"))
print(f"Benchmark flat rule (identical in every run): EUR {primary_results[FLAT]['group']['cost_eur']:,.0f}, "
      f"{primary_results[FLAT]['group']['walks_per_1000']:.2f} walks per 1,000.")

,,cost (EUR),empty rooms,walked guests,"walks per 1,000",guardrail,saving vs flat rule (EUR)
run,method,,,,,,
1 (frozen before any test result),"CatBoost, fitted on train","3,365,371","7,322",210,8.42,met,"-150,459"
2,"XGBoost, fitted on train",-,-,-,-,not run (superseded),-
"3 (after run 1's result), primary","CatBoost, refitted on train + val","2,804,174","5,889",327,13.11,BREACHED,"+410,738"
"3 (after run 1's result), also reported","XGBoost, refitted on train + val","2,799,388","5,883",326,13.07,BREACHED,"+415,524"


Benchmark flat rule (identical in every run): EUR 3,214,912, 7.18 walks per 1,000.


## Step 9: save the results and the run record

In [9]:
primary_table.to_csv(model_dir / "test_run3_results_primary.csv")
secondary_table.to_csv(model_dir / "test_run3_results_secondary.csv")
accuracy_table.to_csv(model_dir / "test_run3_accuracy.csv")
all_runs.to_csv(model_dir / "test_all_runs.csv")
run_record_path.write_text(json.dumps({
    "freeze_file": "models/frozen_choice_run3_train_val.json", "freeze_sha256": freeze_fingerprint,
    "predictions_sha256": predictions_sha,
    "primary": {m: primary_results[m]["group"] for m in [*MODELS, FLAT, FLAT_FAIR]},
}, indent=2) + "\n", encoding="utf-8")
for name in ["test_run3_predictions.csv", "test_run3_results_primary.csv", "test_run3_results_secondary.csv",
             "test_run3_accuracy.csv", "test_all_runs.csv", "test_run3_record.json"]:
    print(f"Saved data/models/{name}")

Saved data/models/test_run3_predictions.csv
Saved data/models/test_run3_results_primary.csv
Saved data/models/test_run3_results_secondary.csv
Saved data/models/test_run3_accuracy.csv
Saved data/models/test_all_runs.csv
Saved data/models/test_run3_record.json


## Step 10: the verdict in plain words (generated from the numbers above)

In [10]:
flat = primary_results[FLAT]["group"]
print(f"Disclosure: run 3 was decided after run 1's test result (run 1 CatBoost: EUR {run1['saving_eur']:+,.0f} vs the flat rule).")
for m in MODELS:
    r = primary_results[m]["group"]
    saving = flat["cost_eur"] - r["cost_eur"]
    print(f"- {m} (refitted on train + val{', PRIMARY' if m == PRIMARY else ''}): accuracy "
          f"{accuracy_table.loc[m, 'accuracy (50% cut-off)']:.1%}, ROC AUC {accuracy_table.loc[m, 'ROC AUC']:.3f}; "
          f"cost EUR {r['cost_eur']:,.0f} vs flat rule EUR {flat['cost_eur']:,.0f} -> "
          + (f"saving EUR {saving:,.0f} ({saving / flat['cost_eur']:.1%})" if saving > 0 else f"EUR {-saving:,.0f} MORE, no saving")
          + f"; {r['empty_rooms']:,} empty rooms, {r['walked_guests']:,} walked guests, {r['walks_per_1000']:.2f} walks per 1,000 -> "
          f"guardrail {'MET' if r['ceiling_met'] else 'BREACHED'}.")
fair = primary_results[FLAT_FAIR]["group"]
print(f"- Fairness check, flat rule with train + val rates: EUR {fair['cost_eur']:,.0f}, {fair['walks_per_1000']:.2f} walks per 1,000 "
      f"-> guardrail {'met' if fair['ceiling_met'] else 'BREACHED'}.")
print(f"- Accuracy of predicting 'nobody cancels': {1 - y_test.mean():.1%}.")

Disclosure: run 3 was decided after run 1's test result (run 1 CatBoost: EUR -150,459 vs the flat rule).
- CatBoost (refitted on train + val, PRIMARY): accuracy 63.9%, ROC AUC 0.687; cost EUR 2,804,174 vs flat rule EUR 3,214,912 -> saving EUR 410,738 (12.8%); 5,889 empty rooms, 327 walked guests, 13.11 walks per 1,000 -> guardrail BREACHED.
- XGBoost (refitted on train + val): accuracy 63.5%, ROC AUC 0.690; cost EUR 2,799,388 vs flat rule EUR 3,214,912 -> saving EUR 415,524 (12.9%); 5,883 empty rooms, 326 walked guests, 13.07 walks per 1,000 -> guardrail BREACHED.
- Fairness check, flat rule with train + val rates: EUR 3,193,614, 8.10 walks per 1,000 -> guardrail met.
- Accuracy of predicting 'nobody cancels': 61.3%.


## What these results do and do not establish

- **Run 3 is test-informed.** It was decided after run 1's result, so it must be reported with run 1 (Step 8), and it cannot stand alone as a clean proof result.
- **The buffers were chosen for the train-only models.** The refitted models expect more cancellations (notebook 15, Step 5), so the same buffer oversells more. Whatever happens to the walks here partly reflects that mismatch, which no data was left to correct.
- **The flat rule with train rates is the contract's benchmark**, kept identical across runs. The train + val version is a fairness check only.
- **Historical, final-book replay** (project rule 7), **four provisional inputs** with unverified timing, and **no validation of the vendor's 95% claim**.
- **Hotel walk rates are reported, not judged**: the client has given only a group-wide ceiling (project rule 6).